# TESDA multilingual learner matcher · v5 MVP
Run all cells on a standard **CPU** runtime. This small continuation learns a projection on the frozen v4 encoder. Public job-title labels and catalog passages are embedded in this notebook. The run compares v4 and v5 on held-out title groups and exports the candidate. The MVP first reconstructs the demand-trained v5, then continues on catalog-labeled learner goals, qualifications, jobs and skills. Regional examples include synthetic drafts. Download the archive before disconnecting. No Drive mount or API key is needed.

In [ ]:
#@title Install the encoder and exporter (use Colab's installed PyTorch)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fastembed==0.8.1', 'onnxruntime==1.31.0', 'onnx==1.23.2'], check=True)

In [ ]:
#@title Prepare public demand data and fetch the verified v4 parent
from pathlib import Path
import base64, gzip
ROOT = Path('/content/tesda-v5'); ROOT.mkdir(exist_ok=True)
(ROOT/'demand.json').write_bytes(gzip.decompress(base64.b64decode('...')))
(ROOT/'v5.py').write_text('"""Train a shared linear projection on the frozen v4 encoder; usable standalone in Colab."""\nimport argparse\nimport hashlib\nimport json\nimport shutil\nfrom pathlib import Path\n\nimport numpy as np\n\nV4_SHA = "76cbab6ba7b27672b1f3b5e0b48260b5a4bc0d5d942d4ccaeb625948ab2e5600"\n\n\ndef digest(path):\n    with path.open("rb") as f:\n        return hashlib.file_digest(f, "sha256").hexdigest()\n\n\ndef vectors(folder, queries, passages):\n    from fastembed import TextEmbedding\n    from fastembed.common.model_description import ModelSource, PoolingType\n\n    name = "tesda-track/" + folder.name\n    if not any(m["model"] == name for m in TextEmbedding.list_supported_models()):\n        TextEmbedding.add_custom_model(model=name, pooling=PoolingType.MEAN, normalization=True,\n                                       sources=ModelSource(hf=name), dim=384, model_file="onnx/model.onnx")\n    encoder = TextEmbedding(name, specific_model_path=str(folder), threads=4)\n\n    def encode(prefix, texts):\n        return np.stack(list(encoder.embed([prefix + t for t in texts], batch_size=8))).astype(np.float32)\n\n    return encode("query: ", queries), encode("passage: ", passages)\n\n\ndef unit(x):\n    return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), 1e-12)\n\n\ndef fit(q, p, targets, counts):\n    import torch\n    from torch.nn import functional as F\n\n    torch.set_num_threads(2)\n    # Small full-batch problem even on CPU. Identity preserves v4 exactly at step zero.\n    q, p, targets = map(torch.from_numpy, (q, p, targets))\n    weights = torch.tensor(1 + np.minimum(np.log1p(counts), 3), dtype=q.dtype)\n    identity = torch.eye(p.shape[1])\n    delta = torch.zeros_like(identity, requires_grad=True)\n    opt = torch.optim.SGD([delta], lr=0.12)\n    history = []\n    for step in range(41):\n        scores = 30 * F.normalize(q @ (identity + delta)) @ F.normalize(p @ (identity + delta)).T\n        # Give each catalog owner positive mass, including when a title has several owners.\n        ce = -(targets * scores.log_softmax(dim=1)).sum(dim=1)\n        loss = (ce * weights).sum() / weights.sum()\n        history.append(loss.item())\n        if step < 40:\n            opt.zero_grad()\n            (loss + 0.04 * delta.square().sum()).backward()\n            opt.step()\n    return (identity + delta.detach()).numpy(), history\n\n\ndef export(base, output, matrix):\n    import onnx\n    from onnx import helper, numpy_helper\n\n    model = onnx.load(base / "onnx/model.onnx")\n    graph = model.graph\n    assert len(graph.output) == 1\n    hidden = graph.output[0].name\n    graph.initializer.append(numpy_helper.from_array(matrix, "v5_projection"))\n    # Linear projection commutes with fastembed\'s masked mean. L2 normalization happens last.\n    graph.node.append(helper.make_node("MatMul", [hidden, "v5_projection"], ["v5_hidden"]))\n    graph.output[0].name = "v5_hidden"\n    (output / "onnx").mkdir(parents=True)\n    onnx.save(model, output / "onnx/model.onnx")\n    onnx.checker.check_model(str(output / "onnx/model.onnx"))\n    for name in ("config.json", "tokenizer.json", "tokenizer_config.json", "calibration.json"):\n        shutil.copyfile(base / name, output / name)\n\n\ndef rank_report(q, p, mask):\n    rank = np.argsort(-(unit(q) @ unit(p).T), axis=1, kind="stable")\n    correct = np.take_along_axis(mask, rank, axis=1)\n    pos = np.argmax(correct, axis=1) + 1\n    return {"titles": len(pos), "top1_hits": int(sum(pos == 1)), "top3_hits": int(sum(pos <= 3)),\n            "mrr": float(np.mean(1 / pos))}\n\n\ndef train(base, output, data_path):\n    if output.exists():\n        raise FileExistsError(f"Choose a new output folder: {output}")\n    if digest(base / "onnx/model.onnx") != V4_SHA:\n        raise ValueError("Wrong v4 parent ONNX")\n    data = json.loads(data_path.read_text(encoding="utf-8"))\n    rows, catalog = data["rows"], data["catalog"]\n    queries = [r["title"] for r in rows]\n    texts = [c["passage"] for c in catalog]\n    train_idx = np.array([r["split"] == "train" for r in rows])\n    assert set(r["group"] for r in rows if r["split"] == "train").isdisjoint(\n        r["group"] for r in rows if r["split"] == "held")\n    labels = np.array([[c["name"] in r["labels"] for c in catalog] for r in rows])\n    assert labels.any(axis=1).all() and train_idx.any() and (~train_idx).any()\n    profile = data.get("profile", "v5-demand")\n    print(f"Encoding {len(texts)} catalog passages and {len(queries)} queries for {profile}", flush=True)\n    q, p = vectors(base, queries, texts)\n    positives = labels[train_idx].astype(np.float32)\n    counts = np.array([r["sources"].get("postings", 0) for r in rows])[train_idx]\n    targets = positives / positives.sum(axis=1, keepdims=True)\n    if profile == "v5-mvp":\n        demand = np.array(["lang" not in r for r in rows])[train_idx]\n        prior, _ = fit(q[train_idx][demand], p, targets[demand], counts[demand])\n        # Reconstruct the original v5 without a second model download. Projection before\n        # pooling is linear, so the two learned matrices can be exported as one.\n        revised, history = fit(unit(q[train_idx] @ prior), unit(p @ prior), targets, counts)\n        matrix = prior @ revised\n    else:\n        matrix, history = fit(q[train_idx], p, targets, counts)\n    export(base, output, matrix)\n    # Verify actual serving vectors in both modes, with unequal passage lengths.\n    actual_q, actual_p = vectors(output, queries[:5], texts[:5])\n    parity = float(min((a * unit(b @ matrix)).sum(axis=1).min()\n                       for a, b in [(actual_q, q[:5]), (actual_p, p[:5])]))\n    if not np.isfinite(parity) or parity < 0.999:\n        raise ValueError(f"Export parity failed: {parity}")\n    held = ~train_idx\n    report = {"profile": profile, "parent_sha256": V4_SHA, "dataset_sha256": digest(data_path),\n              "data": data["audit"], "train_titles": int(train_idx.sum()), "steps": len(history) - 1,\n              "train_loss": [history[0], history[-1]], "parity_cosine": parity,\n              "held_v4": rank_report(q[held], p, labels[held]),\n              "held_v5": rank_report(q[held] @ matrix, p @ matrix, labels[held])}\n    if profile == "v5-mvp":\n        report["warm_start"] = {"profile": "v5-demand", "steps": 40, "train_titles": int(demand.sum()),\n                                "held": rank_report(q[held] @ prior, p @ prior, labels[held])}\n    (output / "training.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    print(json.dumps(report, indent=2))\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for arg in ("base", "output", "data"):\n        parser.add_argument("--" + arg, required=True, type=Path)\n    args = parser.parse_args()\n    train(args.base, args.output, args.data)\n')
from huggingface_hub import snapshot_download
base = ROOT/'tesda-e5-small-v4'
snapshot_download('Nyanta19/tesda-e5-small-v4', local_dir=base, allow_patterns=['config.json','tokenizer.json','tokenizer_config.json','calibration.json','onnx/model.onnx'])

In [ ]:
#@title Train v5, score held-out demand titles, and verify the serving export
from datetime import datetime, timezone
out = ROOT/datetime.now(timezone.utc).strftime('run-%Y%m%d-%H%M%S')/'tesda-e5-small-v5-mvp'
run = subprocess.run([sys.executable, '-u', str(ROOT/'v5.py'), '--base', str(base), '--output', str(out), '--data', str(ROOT/'demand.json')], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(run.stdout)
run.check_returncode()

In [ ]:
#@title Download the trained v5 candidate
import shutil
from google.colab import files
archive = shutil.make_archive(str(out), 'zip', root_dir=out.parent, base_dir=out.name)
print('Download the candidate and training.json:', archive)
files.download(archive)